# Phishing Detection System Using Machine Learning

**Project Overview:**  
This notebook presents a complete end-to-end pipeline for phishing URL detection using supervised machine learning algorithms. Four classifiers are trained and compared: Logistic Regression, Support Vector Machine (SVM), Decision Tree, and Random Forest.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Display settings
pd.set_option('display.max_columns', 35)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Libraries loaded successfully.')

## 2. Load Dataset

The dataset contains 11,055 instances with 30 URL-based features extracted from phishing and legitimate websites. Sources: PhishTank, OpenPhish, and UCI Machine Learning Repository.

In [ ]:
# Load the phishing dataset
df = pd.read_csv('../data/phishing_dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'Number of features: {df.shape[1] - 1}')
print(f'Number of samples: {df.shape[0]}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset info
print('=== Dataset Info ===')
print(df.info())
print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

In [ ]:
# Class distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

class_counts = df['Result'].value_counts()
labels = ['Phishing (-1)', 'Legitimate (1)']
colors = ['#f87171', '#34d399']

bars = ax.bar(labels, class_counts.values, color=colors, edgecolor='white', width=0.5)

for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{count}', ha='center', fontsize=13, fontweight='bold')

ax.set_title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (top features)
fig, ax = plt.subplots(figsize=(16, 12))

corr = df.corr()
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)

ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Preparation

In [ ]:
# Separate features and labels
X = df.drop('Result', axis=1)
y = df['Result']

print(f'Features shape: {X.shape}')
print(f'Labels shape:   {y.shape}')
print(f'\nFeature columns: {list(X.columns)}')

In [ ]:
# Split into training and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

## 5. Model Training and Evaluation

Four supervised classification algorithms are trained and evaluated:
1. Logistic Regression
2. Support Vector Machine (SVM)
3. Decision Tree
4. Random Forest

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Support Vector Machine': SVC(kernel='rbf', random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

# Train and evaluate each model
results = []

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    acc  = accuracy_score(y_test, preds) * 100
    prec = precision_score(y_test, preds, pos_label=1, zero_division=0) * 100
    rec  = recall_score(y_test, preds, pos_label=1, zero_division=0) * 100
    f1   = f1_score(y_test, preds, pos_label=1, zero_division=0) * 100
    
    results.append({
        'Model': name,
        'Accuracy (%)': round(acc, 1),
        'Precision (%)': round(prec, 1),
        'Recall (%)': round(rec, 1),
        'F1-Score (%)': round(f1, 1),
    })
    print(f'Done — Accuracy: {acc:.1f}%')

# Display results table
results_df = pd.DataFrame(results)
print('\n' + '='*65)
print('  TABLE 4.1: PERFORMANCE EVALUATION OF CLASSIFICATION MODELS')
print('='*65)
results_df

## 6. Confusion Matrix — Random Forest Model

In [ ]:
# Generate confusion matrix for Random Forest
rf_model = models['Random Forest']
rf_preds = rf_model.predict(X_test)
cm = confusion_matrix(y_test, rf_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Phishing', 'Legitimate'],
    yticklabels=['Phishing', 'Legitimate'],
    annot_kws={'size': 16, 'fontweight': 'bold'},
    ax=ax
)

ax.set_title('Confusion Matrix — Random Forest Model',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.show()

print(f'True Positives:  {cm[1][1]}')
print(f'True Negatives:  {cm[0][0]}')
print(f'False Positives: {cm[0][1]}')
print(f'False Negatives: {cm[1][0]}')

## 7. Model Accuracy Comparison Chart

In [ ]:
# Model accuracy comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#3498db', '#e74c3c', '#f39c12', '#2ecc71']

bars = ax.barh(
    results_df['Model'],
    results_df['Accuracy (%)'],
    color=colors,
    edgecolor='white',
    height=0.5
)

# Add percentage labels
for bar, acc in zip(bars, results_df['Accuracy (%)']):
    ax.text(
        bar.get_width() - 3,
        bar.get_y() + bar.get_height() / 2,
        f'{acc}%',
        va='center', ha='right',
        fontsize=13, fontweight='bold', color='white'
    )

ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_xlim(80, 100)
plt.tight_layout()
plt.show()

## 8. Detailed Classification Report — Random Forest

In [ ]:
# Full classification report
print('='*55)
print('  CLASSIFICATION REPORT — RANDOM FOREST')
print('='*55)
print(classification_report(y_test, rf_preds,
                            target_names=['Phishing', 'Legitimate']))

## 9. Feature Importance — Random Forest

In [ ]:
# Feature importance from Random Forest
importances = rf_model.feature_importances_
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(
    feature_importance['Feature'],
    feature_importance['Importance'],
    color='#4f8cff',
    edgecolor='white',
    height=0.6
)

ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top 10 most important features
print('\nTop 10 Most Important Features:')
print(feature_importance.tail(10).to_string(index=False))

## 10. Save the Best Model

In [ ]:
import joblib
import os

# Save the Random Forest model
model_path = '../models/random_forest_model.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)
joblib.dump(rf_model, model_path)

print(f'Model saved to: {model_path}')
print(f'Model type: {type(rf_model).__name__}')
print(f'Number of estimators: {rf_model.n_estimators}')

## 11. Test Classification on Sample URLs

In [ ]:
import sys
sys.path.insert(0, '..')
from src.feature_extraction import extract_features_from_url

# Test URLs
test_urls = [
    'http://secure-bank-login.xyz@verify-user.com/account/confirm',
    'https://www.google.com',
    'https://www.bank.com/login',
    'http://192.168.1.1/admin/login.php',
    'https://bit.ly/3xF9abc',
]

print('='*65)
print('  SAMPLE URL CLASSIFICATION RESULTS')
print('='*65)

for url in test_urls:
    features = extract_features_from_url(url)
    prediction = rf_model.predict([features])[0]
    label = 'PHISHING' if prediction in [-1, 0] else 'LEGITIMATE'
    icon = '⚠️' if label == 'PHISHING' else '✅'
    print(f'\n  {icon} {label}')
    print(f'     URL: {url}')

print('\n' + '='*65)

---

**End of Notebook**  
Phishing Detection System Using Machine Learning © 2026